# QuantumFlow on a Colab GPU
Runtime > Change runtime type > **T4 GPU**, then run every cell. Upload the repository as a zip first (or `git clone` it once it is on GitHub).

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
# Finds the uploaded project zip (any name starting with "quantumflow"), unzips it, and checks it worked.
import glob, os, subprocess
zips = sorted(glob.glob('/content/**/quantumflow*.zip', recursive=True))
print('zip files found:', zips)
assert zips, 'No quantumflow zip found: drag quantumflow.zip into the Files panel on the left and wait for the upload to finish, then rerun this cell.'
subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content'], check=True)
assert os.path.exists('/content/quantumflow/CMakeLists.txt'), 'Unzipped, but /content/quantumflow/CMakeLists.txt is missing'
print('OK: project is at /content/quantumflow')


In [ ]:
%cd /content/quantumflow
!pip -q install pybind11 pyarrow pyyaml QuantLib
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=75 -DPython_EXECUTABLE=$(which python)
!cmake --build build -j

In [ ]:
!ctest --test-dir build --output-on-failure
!python -m pytest tests/test_validation.py -q

In [ ]:
!./build/cuda_bench results

In [ ]:
# Nsight Systems timeline (preinstalled with the CUDA toolkit on Colab)
!nsys profile -o profiles/qf_timeline --force-overwrite true ./build/cuda_bench results > /dev/null
# Nsight Compute kernel report (may need: !apt-get install -y nsight-compute)
!ncu --set full --kernel-name regex:heston_kernel --launch-count 3 -o profiles/qf_kernel --force-overwrite true ./build/cuda_bench results > /dev/null
!ls -la profiles

In [ ]:
# Re-run the surrogate pipeline with GPU Monte Carlo and regenerate the dashboard
!python python/scripts/pipeline_demo.py
!python python/dashboard/make_plots.py

In [ ]:
# Download results + profiles
!zip -qr qf_gpu_results.zip results profiles
from google.colab import files; files.download('qf_gpu_results.zip')